## Limpieza web_events
Objetivos del notebook: 
- Limpiar los datos de clientes provenientes de la capa bronce.
- Estandarizar y Limpiar los datos. 
- Mandar a cuarentena aquellos datos que requieren revision más profunda. 
- Cargar los datos en la capa plata. 

### Resumen Columnas
| Campo | Descripción |
|---|---|
| `anonymous_id`| Cookie/dispositivo |
| `customer_id` | `customer_id` solo existe si hay sesión iniciada |
| `device` | struct `{browser, os, type, ua}` (`ua` solo aparece cuando el SDK no reconoce el agente) |
| `event_id` | Identificador del evento (los reintentos del SDK lo repiten) |
| `event_ts`| Cuándo ocurrió en el dispositivo y cuándo llegó al servidor (texto ISO-8601, UTC) |
| `event_type` | `page_view`, `search`, `product_view`, `add_to_cart`, `begin_checkout`, `purchase` |
| `geo` | Struct `{country}` |
| `page` | Struct `{referrer, url}` |
| `platform` | `web`/`ios`/`android` |
| `props` | Struct según el evento: `items`,`order_id`, `product_id`, `quantity`, `results`,`search_term`|
| `received_at` | Cuándo ocurrió en el dispositivo        |
| `utm` | Struct `{campaign, medium, source }`; solo viene en el **primer evento** de una visita con campaña |
| `dlh_hash_web_events` | Hash de web_events generado por todas las columnas sin el prefijo `dlh_`     |
| `dlh_carga_id`     | Identificador de la carga a la que pertenece.             |
| `dlh_fecha_ingesta_bronce`| Fecha de ingesta en la capa bronce                 |
| `dlh_fecha_ingesta`| Fecha de ingesta en la capa plata                         |

In [0]:
%load_ext autoreload
%autoreload 2

## Librerias y datos

In [0]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "01-funciones-compartidas"))

from pyspark.sql import functions as F, types as T, Window as W
from delta.tables import DeltaTable
from funciones_limpieza import limpiar_texto, estandarizar_columnas_str, fecha_de_corte_data, generar_hash
from funciones_conexion import obtener_ruta
from funciones_actualizacion_marcas import obtener_ventana_fechas_carga_id, actualizar_fechas_id_carga

## Ingesta de datos

In [0]:
# 1. Lectura de la tabla de Bronce
schema = "bronce"
tabla = "web_events"
schema_actualizacion = "plata"
bronce_eventos_web_t = spark.table(
    "dlh_nova_shop."+schema+"."+tabla
)

# 2. Valores de control de fechas y carga_id 
ultima_fecha_actualizacion, maxima_fecha_bronce,resumen_carga,ventana_carga_id, bronce_eventos_web_t= obtener_ventana_fechas_carga_id(
    spark= spark,
    tabla_df = bronce_eventos_web_t,
    nombre_tabla=tabla,
    nombre_schema_tabla_actualizacion= schema_actualizacion
    )
# 3. Resumen de valores 
print(f"Watermark anterior: {ultima_fecha_actualizacion}")
print(f"Fecha máxima de ingesta en bronce: {maxima_fecha_bronce}")
print(f"Registros a procesar en esta carga incremental: {bronce_eventos_web_t.count()}")
print(f"Cargas a reemplazar (REPLACE WHERE): {ventana_carga_id}")

## Separacion eventos web corruptos y funcionales

In [0]:
# No funciona porque estoy en un modo serverless
# bronce_eventos_web.cache()
bronce_eventos_web_t.write \
        .format("delta") \
        .mode("overwrite") \
        .option('delta.feature.allowColumnDefaults','supported') \
        .saveAsTable(
            "dlh_nova_shop.plata.web_events_lectura"
        )
registros_leidos = spark.read.table("dlh_nova_shop.plata.web_events_lectura")
bronce_eventos_web = registros_leidos.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
C_events_corrupt = registros_leidos.filter(F.col("_corrupt_record").isNotNull()).withColumn(
    "RAZON_CUARENTENA",
    F.lit("REGISTRO DE EVENTO WEB CORRUPTO")
)

## Columnas de ayuda

In [0]:
plata_customers = spark.read.table("dlh_nova_shop.plata.customers").select("customer_id")

In [0]:
plata_products = spark.read.table("dlh_nova_shop.plata.products").select("product_id")

In [0]:
plata_orders = spark.read.table("dlh_nova_shop.plata.sales_lines").select("order_id")

## Limpieza web_events tipos de datos

### Limpieza anonymous_id

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "anonymous_id", 
    F.lower(F.trim(F.col("anonymous_id")))
).withColumn(
    "anonymous_id",
    F.when(
        F.col("anonymous_id").rlike(r"^a-[0-9a-z]{12}$"),
        F.col("anonymous_id")
    ).otherwise(
        "N/A"
    )
)

### Limpieza customer_id

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "customer_id", 
    F.upper(F.trim(F.col("customer_id")))
    ).withColumn(
        "customer_id",
        F.when(
            F.col("customer_id").rlike(r"^C\d{6}$"),
            F.col("customer_id")
        )
        .otherwise("N/A")
    )

### Limpieza device

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "device",
    F.col("device").withField(
        "browser",
        limpiar_texto(F.col("device.browser"))
    ).withField(
        "os",
        F.when( 
            (F.lower(F.trim(F.col("device.os"))).isin(['n/a','null','','-']))
            | (F.col("device.os").isNull()),
            'N/A'
        ).when(
            (F.lower(F.trim(F.col("device.os"))) == "macos"),
            'macOS'
        ).otherwise(
            F.initcap(F.trim(F.col("device.os")))
        )
    ).withField(
        "type",
        limpiar_texto(F.col("device.type"))
    ).withField(
        "ua",
        limpiar_texto(F.col("device.ua"))
    )
)

### Limpieza event_id

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "event_id",
    F.lower(F.trim(F.col("event_id")))
).withColumn(  
    "event_id",
    F.when(
        F.col("event_id").rlike(r"^e-[0-9a-z]{16}$"),
        F.col("event_id")
    ).otherwise(
        "N/A"
    )
)

### Limpieza event_ts

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "event_ts",
    F.try_to_timestamp(F.col("event_ts"), F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX"))
)

### Limpieza event_type

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "event_type",
    F.regexp_replace(
        F.col("event_type"),
        "_",
        " "
    )
).withColumn(
    "event_type",
    limpiar_texto(F.col("event_type"))
)

### Limpieza geo

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "geo",
    F.col("geo").withField(
        "country",
        F.when( 
            (F.lower(F.trim(F.col("geo.country"))).isin(['n/a','null','','-']))
            | (F.col("geo.country").isNull()),
            'N/A'
        ).otherwise(
            F.upper(F.trim(F.col("geo.country")))
        )
    )
)

### Limpieza page

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "page",
    F.col("page").withField(
        "referrer",
        F.when( 
            (F.lower(F.trim(F.col("page.referrer"))).isin(['n/a','null','','-']))
            | (F.col("page.referrer").isNull()),
            'N/A'
        ).otherwise(
            F.lower(F.trim(F.col("page.referrer")))
        )
    ).withField(
        "url",
        F.when( 
            (F.lower(F.trim(F.col("page.url"))).isin(['n/a','null','','-']))
            | (F.col("page.url").isNull()),
            'N/A'
        ).otherwise(
            F.lower(F.trim(F.col("page.url")))
        )
    )
)

### Limpieza platform

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "platform",
    limpiar_texto(F.col("platform"))
)

### Limpieza props

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "props",
    F.col("props").withField(
        "items",
        F.abs(F.col("props.items"))
    ).withField(
        "order_id",
        F.when(
            F.upper(F.trim(F.col("props.order_id"))).rlike(r"^O\d{8}$"),
            F.col("props.order_id")
        )
        .otherwise("N/A")
    ).withField(
        "product_id",
        F.when(
            F.upper(F.trim(F.col("props.product_id"))).rlike(r"^P\d{4}$"),
            F.col("props.product_id")
        )
        .otherwise("N/A")
    ).withField(
        "quantity",
        F.abs(F.col("props.quantity"))
    ).withField(
        "results",
        F.abs(F.col("props.results"))
    ).withField(
        "search_term",
        limpiar_texto(F.col("props.search_term"))
    )
)

### Limpieza received_at


In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "received_at",
    F.try_to_timestamp(F.col("received_at"), F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX"))
)

### Limpieza utm

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "utm",
    F.col("utm").withField(
        "campaign",
        limpiar_texto(F.regexp_replace(F.col("utm.campaign"),"_"," "))
    ).withField(
        "medium",
        limpiar_texto(F.regexp_replace(F.col("utm.medium"),"_"," "))
    ).withField(
        "source",
        limpiar_texto(F.regexp_replace(F.col("utm.source"),"_"," "))
    )
)

### Elimninacion de duplicados

In [0]:
bronce_eventos_web = bronce_eventos_web.dropDuplicates()

## Limpieza web_events logica referencial y fechas

In [0]:
bronce_eventos_web = bronce_eventos_web.withColumn(
    "fecha_de_corte",
    fecha_de_corte_data()
)

In [0]:
C_sin_event_id = bronce_eventos_web.filter(
    F.col("event_id") == "N/A"
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("event_id NULL")
)
bronce_eventos_web = bronce_eventos_web.filter(
    F.col("event_id") != "N/A"
)

### Integridad Referencial

In [0]:
# Para la IA/Persona que lea esto, la verificacion de la integridad referencial va a ser un poco diferente, porque, no necesariamente si el evento no tiene un customer_id, quiere decir que sea invalido, en este caso puede ser, porque, no tiene una sesion conectada 
# Los clientes que no tengan customer_id N/A y ademas, que no esten en la tabla plata_customer, se van a cuarentena
C_sin_customer_id_validos = bronce_eventos_web.filter(
    F.col("customer_id") != "N/A"
).join(
    how="leftanti",
    other=plata_customers,
    on= "customer_id",
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("customer_id NO EXISTE EN plata_customers")
)
# Los customer_id, que no esten en la tabla donde estan los ids invalidos 
bronce_eventos_web = bronce_eventos_web.join(
    how="leftanti",
    other= C_sin_customer_id_validos,
    on = "customer_id"
)

In [0]:
# Los ventos tipo "Pucharse", que no tengan un order_id valido, se van a cuarentena
eventos_order_id = ["Purchase"]
C_sin_order_id_validos = bronce_eventos_web.filter(
    (F.col("event_type").isin(eventos_order_id))
)
C_sin_order_id_validos = C_sin_order_id_validos.join(
    how="leftanti",
    other= plata_orders,
    on = plata_orders["order_id"]== C_sin_order_id_validos["props.order_id"]
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("order_id NO EXISTENTE EN plata_sales_lines")
)
# Los order_id, que no esten en la tabla donde estan los ids invalidos 
bronce_eventos_web = bronce_eventos_web.join(
    how="leftanti",
    other=C_sin_order_id_validos,
    on= "event_id"
)

In [0]:
# Los ventos tipo "Add to Cart","Product View", que no tengan un product_id valido, se van a cuarentena
eventos_product_id = ["Add to Cart","Product View"]
C_sin_product_id_validos = bronce_eventos_web.filter(
    (F.col("event_type").isin(eventos_product_id))
)

registros = C_sin_product_id_validos.alias("p")
productos = plata_products.alias("pr")
C_sin_product_id_validos = registros.join(
    productos,
    on=F.col("p.props.product_id") == F.col("pr.product_id"),
    how="left_anti"
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("product_id NO EXISTENTE EN plata_products")
)
# Los product_id, que no esten en la tabla donde estan los ids invalidos 
bronce_eventos_web = bronce_eventos_web.join(
    how="leftanti",
    other=C_sin_product_id_validos,
    on= "event_id"
)

### Cuarentena por Fechas

In [0]:
C_sin_event_ts = bronce_eventos_web.filter(
    F.col("event_ts").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("event_ts NULL")
)
bronce_eventos_web = bronce_eventos_web.filter(
    F.col("event_ts").isNotNull()
)

In [0]:
C_sin_received_at = bronce_eventos_web.filter(
    F.col("received_at").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("received_at NULL")
)
bronce_eventos_web = bronce_eventos_web.filter(
    F.col("received_at").isNotNull()
)

In [0]:
C_event_ts_mayor_received_at = bronce_eventos_web.filter(
    F.col("event_ts") > F.col("received_at")
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("event_ts > received_at")
)
bronce_eventos_web = bronce_eventos_web.filter(
    F.col("event_ts") <= F.col("received_at")
)

In [0]:
C_event_ts_futura = bronce_eventos_web.filter(
    F.col("event_ts") > F.col("fecha_de_corte")
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("FECHA FUTURA DE event_ts ")
)
bronce_eventos_web = bronce_eventos_web.filter(
    F.col("event_ts") <= F.col("fecha_de_corte")
)

In [0]:
C_recived_at_futura = bronce_eventos_web.filter(
    F.col("received_at") > F.col("fecha_de_corte")
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("FECHA FUTURA DE received_at ")
)
bronce_eventos_web = bronce_eventos_web.filter(
    F.col("received_at") <= F.col("fecha_de_corte")
)

### Deduplicar Datos

In [0]:
# Para un mismo evento, seleccina el evento con la fecha mas actual
bronce_eventos_web = bronce_eventos_web.withColumn(
    "numero_evento",
    F.row_number().over(
        W.partitionBy(
            F.col("event_id")
        ).orderBy(
            F.col("received_at").desc()
        )
    )
).filter(
    F.col("numero_evento") == 1
).drop("numero_evento")

## Exportando datos

In [0]:
cuarentena_web_events = C_events_corrupt \
    .unionByName(C_sin_event_id,allowMissingColumns=True) \
    .unionByName(C_sin_customer_id_validos, allowMissingColumns=True) \
    .unionByName(C_sin_product_id_validos, allowMissingColumns=True) \
    .unionByName(C_sin_order_id_validos, allowMissingColumns=True) \
    .unionByName(C_sin_event_ts, allowMissingColumns=True) \
    .unionByName(C_sin_received_at, allowMissingColumns=True) \
    .unionByName(C_event_ts_mayor_received_at, allowMissingColumns=True) \
    .unionByName(C_event_ts_futura, allowMissingColumns=True) \
    .unionByName(C_recived_at_futura, allowMissingColumns=True)

# option("path","LINK ADLSG2")
# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana.
# Si la ejecución falla a medias, la re-ejecución reemplaza las mismas filas (sin duplicados)
if ventana_carga_id is not None:
    cuarentena_web_events.write \
        .mode("overwrite") \
        .format("delta") \
        .option("replaceWhere", ventana_carga_id) \
        .option("delta.feature.allowColumnDefaults", "supported") \
        .saveAsTable("dlh_nova_shop.cuarentena.web_events")
else:
    print("Bronce vacio: no hay registros que escribir en cuarentena")

In [0]:
columnas_hash = [
    "anonymous_id","customer_id", "device", "event_id", "event_ts", "event_type", "geo", "page","platform", "props", "received_at", "utm"
]
plata_eventos_web = generar_hash(bronce_eventos_web,columnas_hash,"dlh_hash_web_events")

plata_eventos_web = plata_eventos_web.select(
    F.col("anonymous_id"),
    F.col("customer_id"),
    F.col("device"),
    F.col("event_id"),
    F.col("event_ts"),
    F.col("event_type"),
    F.col("geo"),
    F.col("page"),
    F.col("platform"),
    F.col("props"),
    F.col("received_at"),
    F.col("utm"),
    F.col("dlh_hash_web_events"),
    F.col("dlh_carga_id"),
    F.col("dlh_fecha_ingesta").alias("dlh_fecha_ingesta_bronce"),
    F.current_timestamp().alias("dlh_fecha_ingesta")
)


In [0]:
plata_eventos_web = plata_eventos_web.withColumn(
    "dedup_hash",
    F.row_number().over(
        W.partitionBy(
            F.col("dlh_hash_web_events")
        )
    )
).filter(
    F.col("dedup_hash") == 1
).drop("dedup_hash")

In [0]:
# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana
if ventana_carga_id is not None:
    DeltaTable.forName(spark, "dlh_nova_shop.plata.web_events").alias("recur").merge(
        plata_eventos_web.alias("incre"),
        "recur.dlh_hash_web_events == incre.dlh_hash_web_events"
    ).whenNotMatchedInsertAll().whenMatchedUpdateAll().execute()
else:
    print("Bronce vacio: no hay registros que escribir en plata")

%md
## Actualizacion de tabla de control

In [0]:
actualizar_fechas_id_carga(
    spark= spark,
    resumen_carga= resumen_carga,
    maxima_fecha= maxima_fecha_bronce,
    ultima_fecha_actualizacion = ultima_fecha_actualizacion,
    tabla = tabla,
    schema = schema_actualizacion 
)